# Hi-EF Phase 1 validation-only pilot

Train B1 context (clips I--II) and B2 full (clips I--III) on the frozen source-folder-held-out split. Checkpoints are selected only by validation UAR (validation loss breaks ties). This notebook does not evaluate the test partition.

In [ ]:
from pathlib import Path
import json
import subprocess

REPO = Path('/kaggle/working/hi-ef-materials')
FEATURES = Path('/kaggle/input/datasets/ptrnghieu/hi-ef-features-v2')
OUTPUT = Path('/kaggle/working/phase1_validation')
if (REPO / '.git').exists():
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', 'experiments'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', 'experiments'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'experiments'], check=True)
else:
    subprocess.run([
        'git', 'clone', '--branch', 'experiments', '--single-branch',
        'https://github.com/ptrnghieu/hi-ef-materials.git', str(REPO)
    ], check=True)
assert (FEATURES / '01_00059.pt').exists()
print('Ready at commit:', subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip())

In [ ]:
common = [
    'python', str(REPO / 'experiments/train_baselines.py'),
    '--manifest', str(REPO / 'experiments/manifests/source_folder_split_seed42.csv'),
    '--features-dir', str(FEATURES),
    '--face-pooling', 'masked',
    '--seed', '42',
    '--epochs', '50',
    '--batch-size', '32',
    '--workers', '2',
    '--learning-rate', '1e-4',
    '--weight-decay', '1e-5',
    '--patience', '8',
    '--d-model', '512',
    '--temporal-layers', '2',
    '--inter-layers', '2'
]

In [ ]:
for model in ('context', 'full'):
    run_dir = OUTPUT / f'{model}_seed42'
    command = common + ['--model', model, '--output-dir', str(run_dir)]
    print('Running', model, '->', run_dir, flush=True)
    subprocess.run(command, check=True)

In [ ]:
summary = {}
for model in ('context', 'full'):
    run_dir = OUTPUT / f'{model}_seed42'
    metrics = json.loads((run_dir / 'metrics.json').read_text())
    config = json.loads((run_dir / 'config.json').read_text())
    assert metrics['test'] is None
    assert config['test_evaluation_requested'] is False
    summary[model] = {
        'best_epoch': metrics['best_epoch'],
        'validation': metrics['validation'],
        'num_parameters': config['num_parameters'],
    }
print(json.dumps(summary, indent=2))
(OUTPUT / 'validation_summary_seed42.json').write_text(json.dumps(summary, indent=2) + '\n')

After the run, use **Save Version** with **Save & Run All** so Kaggle preserves `/kaggle/working/phase1_validation`, including checkpoints, histories, validation logits, and the summary. Do not add `--evaluate-test` yet.